In [ ]:
#@markdown We implemented some functions to visualize the hand landmark detection results. <br/> Run the following cell to activate the functions.
import cv2 
import mediapipe as mp
import numpy as np

I0000 00:00:1786989161.632302  142594 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786989161.632814  142594 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786989161.687614  142594 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786989163.570382  142594 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_E

In [ ]:
img = cv2.imread("Hand.jpg") # Carga la imagen en modo BGR

[ WARN:0@4.205] global loadsave.cpp:278 findDecoder imread_('Hand.jpg'): can't open/read file: check file path/integrity


In [ ]:
# Módulos de MediaPipe para la gestión de conexiones, dibujo y estilos predeterminados
mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

# Constantes de configuración visual para el texto y las etiquetas
MARGIN = 10  # Margen en píxeles para separar el texto de la caja delimitadora
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54) # Color verde vibrante en formato RGB

def draw_landmarks_on_image(rgb_image, detection_result):
  # Extrae la lista de puntos clave (landmarks) y la lateralidad (mano izquierda/derecha)
  hand_landmarks_list = detection_result.hand_landmarks
  handedness_list = detection_result.handedness
  
  # Crea una copia de la imagen original en formato numpy para no modificar la fuente directamente
  annotated_image = np.copy(rgb_image)

  # Itera a través de cada una de las manos detectadas en la imagen
  for idx in range(len(hand_landmarks_list)):
    hand_landmarks = hand_landmarks_list[idx]
    handedness = handedness_list[idx]

    # Dibuja los puntos clave (articulaciones) y las conexiones óseas de la mano
    mp_drawing.draw_landmarks(
      annotated_image,
      hand_landmarks,
      mp_hands.HAND_CONNECTIONS,
      mp_drawing_styles.get_default_hand_landmarks_style(),
      mp_drawing_styles.get_default_hand_connections_style())

    # Calcula la esquina superior izquierda de la mano detectada para ubicar el texto
    height, width, _ = annotated_image.shape
    x_coordinates = [landmark.x for landmark in hand_landmarks]
    y_coordinates = [landmark.y for landmark in hand_landmarks]
    text_x = int(min(x_coordinates) * width)
    text_y = int(min(y_coordinates) * height) - MARGIN

    # Dibuja el texto de la lateralidad (ej. "Left" o "Right") sobre la imagen anotada
    cv2.putText(annotated_image, f"{handedness[0].category_name}",
                (text_x, text_y), cv2.FONT_HERSHEY_DUPLEX,
                FONT_SIZE, HANDEDNESS_TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)

  return annotated_image

In [ ]:
# PASO 1: Importar los módulos necesarios de MediaPipe
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# PASO 2: Configurar y crear el objeto 'HandLandmarker'
# Se define la ruta del modelo entrenado y se especifica el número máximo de manos a detectar (2)
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options,
                                       num_hands=2)
detector = vision.HandLandmarker.create_from_options(options)

# PASO 3: Cargar la imagen de entrada desde el archivo local
image = mp.Image.create_from_file("Human-Hands-2.jpg")

# PASO 4: Realizar la detección de los puntos clave de las manos en la imagen
detection_result = detector.detect(image)

# PASO 5: Procesar el resultado de la detección (en este caso, aplicando la función de visualización)
# image.numpy_view() extrae la matriz de la imagen procesable por la función de dibujo
annotated_image = draw_landmarks_on_image(image.numpy_view(), detection_result)

# Convierte el espacio de color de RGB (usado por MediaPipe) a BGR (requerido por OpenCV para mostrar colores correctos)
bgr_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)

# Muestra la imagen resultante en una ventana estándar de OpenCV
cv2.imshow("Hand Landmarks", bgr_image)
cv2.waitKey(0)          # Pausa la ejecución hasta que el usuario presione cualquier tecla
cv2.destroyAllWindows() # Cierra todas las ventanas abiertas de OpenCV al finalizar

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1786989165.393081  142692 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786989165.407303  142694 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


FileNotFoundError: Can't find file: Human-Hands-2.jpg